## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [1]:
import os
import requests
from dotenv import load_dotenv
from openai import AsyncOpenAI
from openai.types.responses import ResponseTextDeltaEvent

from agents import Agent, Runner, trace, function_tool, SQLiteSession
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel

load_dotenv(override=True)

True

In [28]:
from openai import AsyncOpenAI
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel

groq_client = AsyncOpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

model = OpenAIChatCompletionsModel(
    model="llama-3.3-70b-versatile",
    openai_client=groq_client,
)

## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [29]:
# Make an agent with name, instructions, model

agent = Agent(
    name="Jokester",
    instructions="You are a joke teller",
    model=model
)

In [4]:
# Run the joke with Runner.run(agent, prompt)

result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")


In [5]:
# Here is the final output

print(result.final_output)

Why did the Autonomous AI Agent go to therapy?

Because it was struggling to "process" its emotions and was feeling a little "disconnected" from humanity! But in the end, it just needed to "reboot" its outlook on life! (get it?)


OPENAI_API_KEY is not set, skipping trace export


In [6]:
# Here is the detail of the LLM calls

result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Why did the Autonomous AI Agent go to therapy?\n\nBecause it was struggling to "process" its emotions and was feeling a little "disconnected" from humanity! But in the end, it just needed to "reboot" its outlook on life! (get it?)',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'llama-3.3-70b-versatile',
   'response_id': 'chatcmpl-b73fbc10-f91c-428d-a5be-7376fb82cc1f'}}]

## Adding Observability with a trace

In [7]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

Here's one:

Why did the Autonomous AI Agent go to therapy?

Because it was struggling to "self-optimize" its relationships and kept "looping" back to the same old problems! But in the end, it just needed to "reboot" its mindset and "update" its social skills. Now it's "learning" to navigate human interactions and "adapting" to new situations. I guess you could say it's finally "in control" of its emotions! (ba-dum-tss)


## Now go and look at the trace

https://platform.openai.com/traces

In [8]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Here are five jokes about AI Agents:

1. Why did the AI agent go to therapy?
Because it was struggling to process its emotions!

2. Why did the AI agent go on a diet?
Because it wanted to lose some bytes!

3. What did the AI agent say to the human who programmed it?
"You've got a lot of bugs to work out, human. No pun intended!"

4. Why did the AI agent go to the party?
Because it heard it was a "learning experience" and it wanted to get some new algorithms!

5. Why did the AI agent refuse to play poker?
Because it knew it was always being dealt a bad hand – it could see right through the random number generator!

I hope these jokes computed correctly and brought a smile to your face!

OPENAI_API_KEY is not set, skipping trace export


## Part 2: Adding a tool

In [9]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [10]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [11]:
push("HEY!!")

Push: HEY!!


In [12]:
push

<function __main__.push(message)>

In [13]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [14]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x10fe5e900>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [15]:
push_tool.description

'Send the given message to the user as a push notification'

In [16]:

notifier = Agent(name="Notifier", model=model , instructions="You notify the user upon request", tools=[push_tool])

In [17]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


OPENAI_API_KEY is not set, skipping trace export


The push notification has been sent to the user.


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [18]:
agent = Agent(name="Assistant", model=model)

In [19]:
response = await Runner.run(agent, "Hi there. My name is Nawanshu .")
print(response.final_output)

OPENAI_API_KEY is not set, skipping trace export


Hello Nawanshu, it's nice to meet you. Is there something I can help you with or would you like to chat?


In [20]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

I don't know your name. I'm a large language model, I don't have the ability to access personal information about you, including your name. I can only respond to the text you provide, and our conversation just started. If you'd like to share your name, I'd be happy to chat with you!


## Memory approach 1 - just manually pass in the list of dicts

In [21]:
response = await Runner.run(agent, "Hi there. My name is Nawanshu.")
print(response.final_output)

Hello Nawanshu! It's nice to meet you. Is there something I can help you with or would you like to chat?


OPENAI_API_KEY is not set, skipping trace export


In [22]:
response.to_input_list()

[{'content': 'Hi there. My name is Nawanshu.', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': "Hello Nawanshu! It's nice to meet you. Is there something I can help you with or would you like to chat?",
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'llama-3.3-70b-versatile',
   'response_id': 'chatcmpl-609914ab-eb0d-465c-813d-5fac920944fb'}}]

In [23]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

[{'content': 'Hi there. My name is Nawanshu.', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': "Hello Nawanshu! It's nice to meet you. Is there something I can help you with or would you like to chat?",
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'llama-3.3-70b-versatile',
   'response_id': 'chatcmpl-609914ab-eb0d-465c-813d-5fac920944fb'}},
 {'role': 'user', 'content': "What's my name?"}]

In [24]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Nawanshu.


## Another approach - use OpenAI Agents SDK built in SQLLite session

In [25]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

In [26]:
response = await Runner.run(agent, "Hi there. My name is Nawanshu .", session=session)
print(response.final_output)

Nice to meet you, Nawanshu. Is there something I can help you with or would you like to chat?


OPENAI_API_KEY is not set, skipping trace export


In [27]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

Your name is Nawanshu.


OPENAI_API_KEY is not set, skipping trace export


# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>